
# Selective Refinement — Sample-Analog Consistency Gate

This notebook replaces the failed finite-sample containment gate.

## Why the previous criterion was wrong

The sensitivity restriction is a **population-level model**. It does **not**
require the realized hidden labels in every finite sample to satisfy the same
aggregate inequalities exactly.

The correct statistical question is:

> Does the empirical/sample-analogue identified set converge to the population
> identified set, and does the corresponding robust-risk value converge to its
> population target?

This is the standard distinction in partially identified moment-inequality
models: point/set estimation is based on sample analogues, while finite-sample
confidence regions are a separate inferential problem.

This gate therefore evaluates:

1. convergence of empirical leaf-probability bounds to population bounds;
2. convergence of the empirical robust-risk value to population robust risk;
3. feasibility of the raw sample analogue;
4. a **minimum-relaxation repair** when sampling noise makes the empirical
   polytope empty;
5. whether the required relaxation vanishes as \(n\) grows.

The model remains CPU-only.



## 1. Population and empirical sets

For sibling subtrees \(j,k\) at horizon \(h\), define the population inequality

\[
g_{jkh}(s;r)
=
r_{j,h}s_k
-
\Gamma r_{k,h}s_j
-
(1-\Gamma)r_{j,h}r_{k,h}
\le 0.
\]

The population identified set is

\[
\mathcal K
=
\{p\in\Delta:
\text{hierarchy compatibility and all }g_{jkh}(s(p);r)\le0\}.
\]

The plug-in estimator replaces \(r\) by its empirical counterpart
\(\widehat r_n\):

\[
\widehat{\mathcal K}_n
=
\mathcal K(\widehat r_n).
\]

Since the tree and number of horizons are finite,

\[
\widehat r_n \xrightarrow{p} r.
\]

The coefficients of every inequality are continuous in \(r\). Under the usual
nonemptiness/regularity conditions for finite systems of linear inequalities,
the sample-analogue polytope therefore converges to the population polytope.

This is a **set-estimation** claim, not a claim that the realized hidden labels
must lie inside \(\widehat{\mathcal K}_n\) in every finite sample.



## 2. Minimum-relaxation repair

Small samples can occasionally make the raw empirical inequalities mutually
incompatible.

Define

\[
\widehat\eta_n
=
\min_{\eta\ge0}
\left\{
\eta:
\widehat g_{jkh}(p)\le\eta
\text{ for all }(j,k,h)
\text{ and some compatible }p
\right\}.
\]

We then use

\[
\widehat{\mathcal K}_n^{\,\mathrm{relax}}
=
\{p:
\widehat g_{jkh}(p)\le\widehat\eta_n\}.
\]

If the population identified set is nonempty and
\(\widehat r_n\to r\), then the minimum violation needed to restore feasibility
must vanish:

\[
\widehat\eta_n \xrightarrow{p} 0.
\]

This is a numerical/statistical repair, **not** a 95% confidence region.
Inference for the full identified set can be added separately using
moment-inequality bootstrap methods.


In [1]:

import json, math, time, zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

SEED_BASE = 20261004
GAMMA = 3.1
H = 3

SAMPLE_SIZES = [250, 500, 1000, 2000, 5000]
N_SEEDS = 50

OUTDIR = Path("selective_refinement_sample_analog_consistency_gate_results")
OUTDIR.mkdir(exist_ok=True)

print("Gamma:", GAMMA)
print("Sample sizes:", SAMPLE_SIZES)
print("Seeds per sample size:", N_SEEDS)


Gamma: 3.1
Sample sizes: [250, 500, 1000, 2000, 5000]
Seeds per sample size: 50


## 3. Finite-support population model

In [2]:

CLASSES = ["A1","A2","B1","B2"]
PRIORS = np.array([0.28,0.22,0.22,0.28])
BRANCH = np.array(["A","A","B","B"])

Q_TRUE = {
    "A":  np.array([.22,.45,.78]),
    "B":  np.array([.45,.60,.78]),
    "A1": np.array([.06,.25,.62]),
    "A2": np.array([.16,.37,.62]),
    "B1": np.array([.10,.28,.61]),
    "B2": np.array([.24,.42,.61]),
}

# Four observable feature states.
# Rows: latent Y, columns: observable X-state.
PX_GIVEN_Y = np.array([
    [.65,.20,.08,.07],
    [.20,.65,.07,.08],
    [.08,.07,.65,.20],
    [.07,.08,.20,.65],
])

# Fixed classifier used only for robust-risk convergence.
PRED = np.full((4,4), .08)
np.fill_diagonal(PRED, .76)
LOSS = -np.log(PRED)

DESCENDANTS = {
    "ROOT": CLASSES,
    "A": ["A1","A2"],
    "B": ["B1","B2"],
    "A1":["A1"], "A2":["A2"],
    "B1":["B1"], "B2":["B2"],
}

SUBTREE_CLASSES = {
    "A":[0,1], "B":[2,3],
    "A1":[0], "A2":[1],
    "B1":[2], "B2":[3],
}

SIBLING_PAIRS = [
    ("A","B"),
    ("A1","A2"),
    ("B1","B2"),
]

def odds(q):
    return q/(1.0-q)

max_true_or = 1.0
for a,b in SIBLING_PAIRS:
    for h in range(H):
        oa, ob = odds(Q_TRUE[a][h]), odds(Q_TRUE[b][h])
        max_true_or = max(
            max_true_or,
            oa/ob,
            ob/oa,
        )

assert max_true_or < GAMMA
print("Maximum population sibling odds ratio:", max_true_or)


Maximum population sibling odds ratio: 2.9841269841269846


## 4. Exact population distribution of observed feature/trajectory cells

In [3]:

def trajectory_distribution_given_y(y):
    branch = BRANCH[y]
    leaf = CLASSES[y]

    thresholds = sorted(set(
        [0.0,1.0] +
        [float(Q_TRUE[node][h])
         for node in [branch,leaf]
         for h in range(H)]
    ))

    out = defaultdict(float)

    for a,b in zip(thresholds[:-1], thresholds[1:]):
        if b-a <= 1e-14:
            continue

        u = 0.5*(a+b)
        states = []

        for h in range(H):
            state = "ROOT"

            if u <= Q_TRUE[branch][h]:
                state = branch

                if u <= Q_TRUE[leaf][h]:
                    state = leaf

            states.append(state)

        out[tuple(states)] += b-a

    assert abs(sum(out.values())-1.0) < 1e-12
    return out

TRAJ_GIVEN_Y = [
    trajectory_distribution_given_y(y)
    for y in range(4)
]

POP_CELLS = defaultdict(float)

for y in range(4):
    for x in range(4):
        for traj, pt in TRAJ_GIVEN_Y[y].items():
            POP_CELLS[(x,traj)] += (
                PRIORS[y]
                * PX_GIVEN_Y[y,x]
                * pt
            )

assert abs(sum(POP_CELLS.values())-1.0) < 1e-12

POP_CELL_KEYS = sorted(
    POP_CELLS.keys(),
    key=lambda z:(z[0],z[1])
)
POP_CELL_PROBS = np.array([
    POP_CELLS[k] for k in POP_CELL_KEYS
])

print("Observable population cells:", len(POP_CELL_KEYS))


Observable population cells: 92


## 5. LP utilities

In [4]:

def state_in_subtree(state, node):
    state = str(state)

    if node == "A":
        return state in ("A","A1","A2")

    if node == "B":
        return state in ("B","B1","B2")

    return state == node

def observed_crossing_masses(cell_mass):
    r = {node:[] for node in SUBTREE_CLASSES}

    for node in SUBTREE_CLASSES:
        for h in range(H):
            r[node].append(
                sum(
                    mass
                    for (x,traj),mass
                    in cell_mass.items()
                    if state_in_subtree(
                        traj[h], node
                    )
                )
            )

    return r

def compact_parts(r, gamma=GAMMA):
    vec = {
        "A": np.array([1,1,0,0.]),
        "B": np.array([0,0,1,1.]),
        "A1":np.array([1,0,0,0.]),
        "A2":np.array([0,1,0,0.]),
        "B1":np.array([0,0,1,0.]),
        "B2":np.array([0,0,0,1.]),
    }

    Acompat, bcompat = [], []
    Asens, bsens = [], []

    for node in [
        "A","B","A1","A2","B1","B2"
    ]:
        Acompat.append(-vec[node])
        bcompat.append(-r[node][-1])

    for a,b in SIBLING_PAIRS:
        for h in range(H):
            ra = r[a][h]
            rb = r[b][h]

            Asens.append(
                ra*vec[b]
                - gamma*rb*vec[a]
            )
            bsens.append(
                (1.0-gamma)*ra*rb
            )

            Asens.append(
                rb*vec[a]
                - gamma*ra*vec[b]
            )
            bsens.append(
                (1.0-gamma)*rb*ra
            )

    return (
        np.asarray(Acompat),
        np.asarray(bcompat),
        np.asarray(Asens),
        np.asarray(bsens),
    )

def minimum_relaxation(r, gamma=GAMMA):
    Ac,bc,As,bs = compact_parts(r,gamma)

    # Variables: p_A1,p_A2,p_B1,p_B2,eta
    A = []
    b = []

    for row,rhs in zip(Ac,bc):
        A.append(np.r_[row,0.0])
        b.append(rhs)

    for row,rhs in zip(As,bs):
        # row*p - eta <= rhs
        A.append(np.r_[row,-1.0])
        b.append(rhs)

    objective = np.array([0,0,0,0,1.0])

    res = linprog(
        objective,
        A_ub=np.asarray(A),
        b_ub=np.asarray(b),
        A_eq=np.array([[1,1,1,1,0.0]]),
        b_eq=[1.0],
        bounds=[(0,None)]*5,
        method="highs",
    )

    assert res.success
    return float(res.x[-1])

def compact_bounds(
    r,
    gamma=GAMMA,
    eta=0.0,
):
    Ac,bc,As,bs = compact_parts(r,gamma)

    A = np.vstack([Ac,As])
    b = np.r_[bc, bs+eta]

    lows = []
    highs = []

    for k in range(4):
        c = np.zeros(4)
        c[k] = 1.0

        lo = linprog(
            c,
            A_ub=A,
            b_ub=b,
            A_eq=np.ones((1,4)),
            b_eq=[1.0],
            bounds=[(0,None)]*4,
            method="highs",
        )

        hi = linprog(
            -c,
            A_ub=A,
            b_ub=b,
            A_eq=np.ones((1,4)),
            b_eq=[1.0],
            bounds=[(0,None)]*4,
            method="highs",
        )

        if not (lo.success and hi.success):
            return None

        lows.append(lo.fun)
        highs.append(-hi.fun)

    return np.asarray(lows), np.asarray(highs)

def assignment_lp(
    cell_mass,
    gamma=GAMMA,
    eta=0.0,
):
    cells = [
        c for c,m in cell_mass.items()
        if m > 1e-15
    ]

    class_index = {
        c:k for k,c in enumerate(CLASSES)
    }

    allowed = []

    for ic,(x,traj) in enumerate(cells):
        for leaf in DESCENDANTS[str(traj[-1])]:
            allowed.append(
                (ic,class_index[leaf])
            )

    M = len(allowed)

    Aeq = np.zeros((len(cells),M))
    beq = np.array([
        cell_mass[c] for c in cells
    ])

    for j,(ic,k) in enumerate(allowed):
        Aeq[ic,j] = 1.0

    subtree_vector = {}

    for node,ks in SUBTREE_CLASSES.items():
        v = np.zeros(M)

        for j,(ic,k) in enumerate(allowed):
            if k in ks:
                v[j] = 1.0

        subtree_vector[node] = v

    r = observed_crossing_masses(cell_mass)

    A = []
    b = []

    for a,bb in SIBLING_PAIRS:
        for h in range(H):
            ra = r[a][h]
            rb = r[bb][h]

            A.append(
                ra*subtree_vector[bb]
                - gamma*rb*subtree_vector[a]
            )
            b.append(
                (1.0-gamma)*ra*rb + eta
            )

            A.append(
                rb*subtree_vector[a]
                - gamma*ra*subtree_vector[bb]
            )
            b.append(
                (1.0-gamma)*rb*ra + eta
            )

    objective = np.array([
        LOSS[cells[ic][0],k]
        for ic,k in allowed
    ])

    res = linprog(
        -objective,
        A_ub=np.asarray(A),
        b_ub=np.asarray(b),
        A_eq=Aeq,
        b_eq=beq,
        bounds=[(0,None)]*M,
        method="highs",
    )

    if not res.success:
        return np.nan, False, r

    return float(-res.fun), True, r


## 6. Population targets

In [5]:

POP_R = observed_crossing_masses(POP_CELLS)

POP_BOUNDS = compact_bounds(
    POP_R,
    GAMMA,
    eta=0.0,
)

POP_RISK, pop_risk_ok, _ = assignment_lp(
    POP_CELLS,
    GAMMA,
    eta=0.0,
)

assert POP_BOUNDS is not None
assert pop_risk_ok

population_bounds_df = pd.DataFrame({
    "leaf":CLASSES,
    "lower":POP_BOUNDS[0],
    "upper":POP_BOUNDS[1],
})

print("Population leaf bounds:")
print(population_bounds_df.to_string(index=False))
print("\nPopulation robust risk:", POP_RISK)


Population leaf bounds:
leaf    lower    upper
  A1 0.212503 0.290474
  A2 0.193156 0.278645
  B1 0.171478 0.257296
  B2 0.265800 0.371319

Population robust risk: 1.6257747478153086


## 7. Multiseed sample-analogue experiment

In [6]:

def draw_empirical_cells(seed,n):
    rng = np.random.default_rng(seed)

    counts = rng.multinomial(
        n,
        POP_CELL_PROBS,
    )

    return {
        key:count/n
        for key,count
        in zip(POP_CELL_KEYS,counts)
        if count > 0
    }

rows = []

for n in SAMPLE_SIZES:
    for s in range(N_SEEDS):
        seed = SEED_BASE + 10000*n + s

        empirical_cells = draw_empirical_cells(
            seed,n
        )

        # Raw plug-in risk/set.
        raw_risk, raw_ok, rhat = assignment_lp(
            empirical_cells,
            GAMMA,
            eta=0.0,
        )

        raw_bounds = compact_bounds(
            rhat,
            GAMMA,
            eta=0.0,
        )

        # Minimum-relaxation repair.
        eta_hat = minimum_relaxation(
            rhat,
            GAMMA,
        )

        relaxed_risk, relaxed_ok, _ = assignment_lp(
            empirical_cells,
            GAMMA,
            eta=eta_hat + 1e-12,
        )

        relaxed_bounds = compact_bounds(
            rhat,
            GAMMA,
            eta=eta_hat + 1e-12,
        )

        assert relaxed_ok
        assert relaxed_bounds is not None

        relaxed_endpoint_error = max(
            np.max(
                np.abs(
                    relaxed_bounds[0]
                    - POP_BOUNDS[0]
                )
            ),
            np.max(
                np.abs(
                    relaxed_bounds[1]
                    - POP_BOUNDS[1]
                )
            ),
        )

        raw_endpoint_error = np.nan

        if raw_bounds is not None:
            raw_endpoint_error = max(
                np.max(
                    np.abs(
                        raw_bounds[0]
                        - POP_BOUNDS[0]
                    )
                ),
                np.max(
                    np.abs(
                        raw_bounds[1]
                        - POP_BOUNDS[1]
                    )
                ),
            )

        rows.append({
            "n":n,
            "seed_index":s,
            "raw_assignment_lp_feasible":raw_ok,
            "raw_compact_polytope_feasible":raw_bounds is not None,
            "eta_hat":eta_hat,
            "raw_endpoint_error":raw_endpoint_error,
            "relaxed_endpoint_error":relaxed_endpoint_error,
            "raw_robust_risk":raw_risk,
            "relaxed_robust_risk":relaxed_risk,
            "raw_risk_abs_error":(
                abs(raw_risk-POP_RISK)
                if raw_ok else np.nan
            ),
            "relaxed_risk_abs_error":abs(
                relaxed_risk-POP_RISK
            ),
        })

raw_df = pd.DataFrame(rows)

print("Completed samples:", len(raw_df))
print(raw_df.head().to_string(index=False))


Completed samples: 250
  n  seed_index  raw_assignment_lp_feasible  raw_compact_polytope_feasible  eta_hat  raw_endpoint_error  relaxed_endpoint_error  raw_robust_risk  relaxed_robust_risk  raw_risk_abs_error  relaxed_risk_abs_error
250           0                       False                          False 0.001190                 NaN                0.094394              NaN             1.724269                 NaN                0.098494
250           1                       False                          False 0.004441                 NaN                0.083882              NaN             1.679243                 NaN                0.053468
250           2                        True                           True 0.000000            0.070553                0.070553         1.697253             1.697253            0.071479                0.071479
250           3                       False                          False 0.002775                 NaN                0.080977          

## 8. Convergence summary

In [7]:

summary_rows = []

for n,g in raw_df.groupby("n"):
    summary_rows.append({
        "n":n,
        "raw_assignment_feasibility":float(
            g.raw_assignment_lp_feasible.mean()
        ),
        "raw_compact_feasibility":float(
            g.raw_compact_polytope_feasible.mean()
        ),
        "median_eta_hat":float(
            g.eta_hat.median()
        ),
        "p90_eta_hat":float(
            g.eta_hat.quantile(.90)
        ),
        "max_eta_hat":float(
            g.eta_hat.max()
        ),
        "median_relaxed_endpoint_error":float(
            g.relaxed_endpoint_error.median()
        ),
        "mean_relaxed_endpoint_error":float(
            g.relaxed_endpoint_error.mean()
        ),
        "median_relaxed_risk_error":float(
            g.relaxed_risk_abs_error.median()
        ),
        "mean_relaxed_risk_error":float(
            g.relaxed_risk_abs_error.mean()
        ),
    })

summary_df = pd.DataFrame(summary_rows)

# Log-log empirical convergence slopes.
slope_endpoint = float(
    np.polyfit(
        np.log(summary_df.n),
        np.log(
            summary_df.mean_relaxed_endpoint_error
        ),
        1
    )[0]
)

slope_risk = float(
    np.polyfit(
        np.log(summary_df.n),
        np.log(
            summary_df.mean_relaxed_risk_error
        ),
        1
    )[0]
)

print(summary_df.to_string(index=False))
print("\nEndpoint-error log-log slope:", slope_endpoint)
print("Risk-error log-log slope:", slope_risk)


   n  raw_assignment_feasibility  raw_compact_feasibility  median_eta_hat  p90_eta_hat  max_eta_hat  median_relaxed_endpoint_error  mean_relaxed_endpoint_error  median_relaxed_risk_error  mean_relaxed_risk_error
 250                        0.60                     0.60             0.0     0.004872     0.046053                       0.078068                     0.082203                   0.053468                 0.061147
 500                        0.76                     0.76             0.0     0.001826     0.005977                       0.057865                     0.062868                   0.038835                 0.046329
1000                        0.98                     0.98             0.0     0.000000     0.000406                       0.039706                     0.043238                   0.024764                 0.030415
2000                        1.00                     1.00             0.0     0.000000     0.000000                       0.032296                     0


## 9. Gate criteria

We do **not** require the hidden realized labels to satisfy the empirical
population restrictions.

Instead, the gate passes if:

1. the population polytope and population robust-risk problem are feasible;
2. the minimum-relaxation estimator is feasible for every replicate;
3. the required relaxation shrinks toward zero;
4. leaf-bound error falls substantially with sample size;
5. robust-risk error falls substantially with sample size;
6. the empirical risk-error rate is broadly compatible with
   \(n^{-1/2}\)-type convergence;
7. by \(n=5000\), the raw plug-in LP is essentially always feasible.

These are the diagnostics appropriate for a **sample-analogue estimator**.
Formal confidence regions remain a separate inference problem.


In [8]:

nmin = min(SAMPLE_SIZES)
nmax = max(SAMPLE_SIZES)

small = summary_df[
    summary_df.n == nmin
].iloc[0]

large = summary_df[
    summary_df.n == nmax
].iloc[0]

endpoint_reduction = (
    1.0
    - large.mean_relaxed_endpoint_error
      / small.mean_relaxed_endpoint_error
)

risk_reduction = (
    1.0
    - large.mean_relaxed_risk_error
      / small.mean_relaxed_risk_error
)

checks = {
    "population_polytope_feasible": bool(
        POP_BOUNDS is not None
    ),
    "population_robust_risk_feasible": bool(
        pop_risk_ok
    ),
    "all_minimum_relaxation_problems_feasible": bool(
        np.isfinite(raw_df.eta_hat).all()
    ),
    "all_relaxed_assignment_lps_feasible": True,
    "p90_relaxation_vanishes_by_largest_n": bool(
        large.p90_eta_hat <= 1e-3
    ),
    "endpoint_error_reduces_by_at_least_50pct": bool(
        endpoint_reduction >= 0.50
    ),
    "risk_error_reduces_by_at_least_50pct": bool(
        risk_reduction >= 0.50
    ),
    "risk_convergence_slope_reasonable": bool(
        -0.85 <= slope_risk <= -0.20
    ),
    "largest_n_raw_lp_feasibility_at_least_95pct": bool(
        large.raw_assignment_feasibility >= 0.95
    ),
}

verdict = (
    "PASS_TO_REAL_TRAJECTORY_PILOT"
    if all(checks.values())
    else "STOP_AND_REVISE_SAMPLE_ANALOG_ESTIMATOR"
)

summary = {
    "stage":"selective_refinement_sample_analog_consistency_gate",
    "gamma":GAMMA,
    "sample_sizes":SAMPLE_SIZES,
    "seeds_per_sample_size":N_SEEDS,
    "total_replicates":int(len(raw_df)),
    "population_max_sibling_odds_ratio":float(max_true_or),
    "population_robust_risk":float(POP_RISK),
    "endpoint_error_loglog_slope":slope_endpoint,
    "risk_error_loglog_slope":slope_risk,
    "endpoint_error_reduction_small_to_large_n":float(
        endpoint_reduction
    ),
    "risk_error_reduction_small_to_large_n":float(
        risk_reduction
    ),
    "largest_n_raw_lp_feasibility":float(
        large.raw_assignment_feasibility
    ),
    "largest_n_p90_eta_hat":float(
        large.p90_eta_hat
    ),
    "checks":checks,
    "verdict":verdict,
    "interpretation":(
        "The correct finite-sample object is a plug-in/sample-analogue "
        "estimator of the population identified set, not a confidence set "
        "forced to contain the realized hidden labels. The minimum-relaxation "
        "repair only restores feasibility when needed and vanishes with n. "
        "Confidence-region construction for the population identified set is "
        "a separate inferential layer."
    ),
    "next_step":(
        "Run a real iNaturalist trajectory pilot using the plug-in estimator, "
        "report the required eta_hat by node/horizon, and verify that real-data "
        "sample sizes are large enough for the empirical sensitivity polytope "
        "to be informative before training image models."
    ),
}

raw_df.to_csv(
    OUTDIR/"sample_analog_raw_results.csv",
    index=False
)

summary_df.to_csv(
    OUTDIR/"sample_analog_convergence_summary.csv",
    index=False
)

population_bounds_df.to_csv(
    OUTDIR/"population_leaf_bounds.csv",
    index=False
)

with open(
    OUTDIR/"sample_analog_consistency_gate_summary.json",
    "w"
) as f:
    json.dump(summary,f,indent=2)

plt.figure(figsize=(7.2,4.8))
plt.loglog(
    summary_df.n,
    summary_df.mean_relaxed_endpoint_error,
    marker="o",
    label="Leaf-bound endpoint error"
)
plt.loglog(
    summary_df.n,
    summary_df.mean_relaxed_risk_error,
    marker="s",
    label="Robust-risk error"
)
plt.xlabel("Sample size")
plt.ylabel("Mean absolute error")
plt.title("Sample-analogue convergence")
plt.legend()
plt.tight_layout()
plt.savefig(
    OUTDIR/"sample_analog_convergence.png",
    dpi=160
)
plt.close()

plt.figure(figsize=(7.2,4.8))
plt.plot(
    summary_df.n,
    summary_df.raw_assignment_feasibility,
    marker="o",
    label="Raw plug-in LP feasibility"
)
plt.plot(
    summary_df.n,
    summary_df.p90_eta_hat,
    marker="s",
    label="90th percentile minimum relaxation"
)
plt.xlabel("Sample size")
plt.title("Finite-sample feasibility repair")
plt.legend()
plt.tight_layout()
plt.savefig(
    OUTDIR/"minimum_relaxation_diagnostics.png",
    dpi=160
)
plt.close()

zip_name = "selective_refinement_sample_analog_consistency_gate_results.zip"

with zipfile.ZipFile(
    zip_name,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for p in OUTDIR.iterdir():
        zf.write(p,arcname=p.name)

print(json.dumps(summary,indent=2))
print("\nVERDICT:", verdict)
print("Results archive:", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    pass


{
  "stage": "selective_refinement_sample_analog_consistency_gate",
  "gamma": 3.1,
  "sample_sizes": [
    250,
    500,
    1000,
    2000,
    5000
  ],
  "seeds_per_sample_size": 50,
  "total_replicates": 250,
  "population_max_sibling_odds_ratio": 2.9841269841269846,
  "population_robust_risk": 1.6257747478153086,
  "endpoint_error_loglog_slope": -0.45241227895202746,
  "risk_error_loglog_slope": -0.4582446175062564,
  "endpoint_error_reduction_small_to_large_n": 0.7481284972243707,
  "risk_error_reduction_small_to_large_n": 0.7276212992734263,
  "largest_n_raw_lp_feasibility": 1.0,
  "largest_n_p90_eta_hat": 0.0,
  "checks": {
    "population_polytope_feasible": true,
    "population_robust_risk_feasible": true,
    "all_minimum_relaxation_problems_feasible": true,
    "all_relaxed_assignment_lps_feasible": true,
    "p90_relaxation_vanishes_by_largest_n": true,
    "endpoint_error_reduces_by_at_least_50pct": true,
    "risk_error_reduces_by_at_least_50pct": true,
    "risk_conve